# m03 lab · Unipolar, bipolar and omnipolar electrograms

*Open EP Mapping Lab. Education only: not for diagnosis, treatment or any clinical decision.*

This lab rebuilds the main results of module m03 in code, so you can see where each one comes from
and break it on purpose.

| Part | What you do | Data |
| --- | --- | --- |
| A | Compute a unipolar electrogram from the volume-conductor integral and find the activation marker | synthetic |
| B | Rotate a bipole and see when the \|cos θ\| rule holds | synthetic |
| C | Treat height and the bipole as spatial filters, with FFTs | synthetic |
| D | Estimate the field from a clique of electrodes; build an omnipole; get conduction velocity from one clique | synthetic |
| E | Look at real bipolar recordings from a decapolar catheter | PhysioNet |

**How to run.** From the course repository: `pip install -r requirements.txt`, then open this notebook in Jupyter.
In Google Colab the first code cell fetches the course code for you. The whole lab runs in about a minute.

Cells that start with `assert` are **checks**: they pass with the code as given. If you change something and a
check fails, that is information, not an error to hide.

Units throughout: mm, ms, conduction velocity in mm/ms (equal to m/s). Potentials are in relative units.

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

# Find the shared model code in labs/python/epml. In Colab, fetch the course repository first.
try:
    import epml
except ImportError:
    if "google.colab" in sys.modules:
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/raghavendrasri/open-ep-mapping-lab"], check=True)
        sys.path.insert(0, "open-ep-mapping-lab/labs/python")
    else:
        sys.path.insert(0, str(Path.cwd() / "labs" / "python"))
    import epml

from epml import electrograms as eg

plt.rcParams.update({"figure.figsize": (8, 3.2), "axes.grid": True, "axes.spines.top": False, "axes.spines.right": False})

## Part A · The unipolar electrogram from first principles

The module's model: a thin sheet of tissue, a planar front with an arctangent voltage profile of width $w$, and a
point electrode at height $h$ in an infinite homogeneous conductor. The membrane current is
$i_m = \beta\,\partial^2 V_m/\partial x^2$, and the potential is the volume-conductor integral

$$\phi(\mathbf r) = \frac{1}{4\pi\sigma}\iint \frac{i_m(\mathbf r')}{\lVert \mathbf r - \mathbf r'\rVert}\,dA'.$$

The module derives a closed form, $U(s) = A\,s/(s^2 + a^2)$ with $a = w + h$, where $s$ is the distance from the front
to the electrode. Here you check it by brute force: sum the integral over a large sheet.

In [ ]:
w, h = 0.3, 1.0                 # front width and electrode height, mm (m03 defaults)
beta = sigma = dV = 1.0         # relative units
A = beta * dV / (2 * np.pi * sigma)
a = w + h

def membrane_current(x, w=w):
    # i_m = beta * d2Vm/dx2 for the arctangent front, front at x = 0
    return beta * dV * 2 * w * x / (np.pi * (x**2 + w**2) ** 2)

def phi_quadrature(x_e, h=h, Y=1e5, X=2000.0, n=400_001):
    # Volume-conductor integral over a sheet |y| < Y. The y-integral is done exactly:
    # integral of dy / sqrt(r^2 + y^2) from -Y to Y = 2 asinh(Y / r).
    x = np.linspace(-X, X, n)
    r = np.sqrt((x_e - x) ** 2 + h**2)
    return np.sum(membrane_current(x) * 2 * np.arcsinh(Y / r)) * (x[1] - x[0]) / (4 * np.pi * sigma)

s = np.linspace(-8, 8, 33)
phi_num = np.array([phi_quadrature(v) for v in s])
phi_closed = eg.unipolar(s, a, A)

fig, ax = plt.subplots()
ax.plot(s, phi_closed, label="closed form  A s / (s² + a²)")
ax.plot(s, phi_num, "o", ms=4, label="direct quadrature")
ax.set(xlabel="s: distance from front to electrode (mm)", ylabel="U (relative units)",
       title="Unipolar potential versus front position")
ax.legend()
plt.show()

err = np.max(np.abs(phi_num - phi_closed)) / np.max(np.abs(phi_closed))
print(f"largest difference: {err:.1e} of the peak")

In [ ]:
assert err < 1e-3, "the closed form should match the integral"

### A2 · From position to time, and the activation marker

At a fixed electrode the front arrives over time: $s = x_e - c\,t$. The classic activation marker is the steepest
downstroke of the unipolar signal. In this ideal model it should land exactly on true activation ($t = 0$).

In [ ]:
c = 1.0                              # m/s = mm/ms
t = np.linspace(-15, 15, 3001)       # ms; the front passes the electrode at t = 0
U = eg.unipolar(-c * t, a, A)
Vm = 0.5 - np.arctan(-c * t / w) / np.pi   # normalized transmembrane voltage under the electrode
dUdt = np.gradient(U, t)
t_steep = t[np.argmin(dUdt)]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 4.6), sharex=True)
ax1.plot(t, Vm, color="gray")
ax1.set(ylabel="Vm (normalized)")
ax2.plot(t, U, label="unipolar electrogram")
ax2.plot(t_steep, U[np.argmin(dUdt)], "o", color="C3", label="steepest downstroke")
for ax in (ax1, ax2):
    ax.axvline(0, ls="--", color="gray")
ax2.set(xlabel="time (ms)", ylabel="U (relative units)")
ax2.legend()
plt.show()
print(f"steepest downstroke at {t_steep:+.3f} ms; true activation at 0 ms")

In [ ]:
assert abs(t_steep) <= t[1] - t[0], "in ideal propagation the steepest downstroke marks activation"

**Your turn.** Before running anything, predict what happens to the peak-to-peak amplitude and the steepest slope if
you double $h$, or double $c$. Then change the values above and rerun. (The module says peak-to-peak scales as
$1/(w+h)$ and the steepest slope as $c/(w+h)^2$.)

### A3 · Amplitude versus height

In [ ]:
heights = np.array([0.25, 0.5, 1, 2, 4])
p2p = np.array([np.ptp(eg.unipolar(-c * t, w + hh, A)) for hh in heights])
slope = np.polyfit(np.log(w + heights), np.log(p2p), 1)[0]

fig, ax = plt.subplots()
ax.loglog(w + heights, p2p, "o-")
ax.set(xlabel="w + h (mm)", ylabel="peak-to-peak (relative units)", title="Unipolar amplitude falls as 1/(w + h)")
plt.show()
print(f"log-log slope: {slope:.3f}   (theory: -1)")

In [ ]:
assert abs(slope + 1) < 0.01

## Part B · Rotating a bipole

A bipolar electrogram is the difference of two unipoles: distal minus proximal. For a planar wave, the two electrodes
sit at $s \pm \tfrac{d}{2}\cos\theta$, where $\theta$ is the angle between the bipole and the direction of propagation.
For a close bipole the module predicts amplitude $\propto |\cos\theta|$. Here you find out how close "close" must be.

In [ ]:
s_long = np.linspace(-80, 80, 16001)
thetas = np.deg2rad(np.arange(0, 181, 5))

def amplitude_vs_angle(d):
    ref = np.ptp(eg.bipolar_planar(s_long, d, 0.0, a, A))
    return np.array([np.ptp(eg.bipolar_planar(s_long, d, th, a, A)) for th in thetas]) / ref

fig, ax = plt.subplots()
for d in (0.2, 2.0, 6.0):
    ax.plot(np.rad2deg(thetas), amplitude_vs_angle(d), label=f"d = {d} mm  (d/a = {d / a:.1f})")
ax.plot(np.rad2deg(thetas), np.abs(np.cos(thetas)), "k--", label="|cos θ|  (close bipole)")
ax.set(xlabel="bipole angle θ (degrees)", ylabel="peak-to-peak / aligned bipole",
       title="Bipolar amplitude versus orientation")
ax.legend()
plt.show()

In [ ]:
ds = np.linspace(0.1, 8, 40)
deviation = np.array([np.max(np.abs(amplitude_vs_angle(d) - np.abs(np.cos(thetas)))) for d in ds])

fig, ax = plt.subplots()
ax.plot(ds / a, deviation)
ax.set(xlabel="spacing / (w + h)", ylabel="largest deviation from |cos θ|",
       title="The |cos θ| rule needs spacing small next to w + h")
plt.show()
dev_small = np.max(np.abs(amplitude_vs_angle(0.2) - np.abs(np.cos(thetas))))
print(f"at d = 0.2 mm the largest deviation is {dev_small:.4f}")

In [ ]:
assert dev_small < 0.01

**Your turn.**

1. In the first plot, the 6-mm bipole records *more* at 60° than when aligned with the wave. Why? (Hint: compare
   the effective spacing $d\cos\theta$ with $a = w + h$, and look at where the two unipoles' peaks fall.)
2. The module claims a bipole is more *local* than a unipole: its amplitude falls as $1/(w+h)^2$ instead of
   $1/(w+h)$. Repeat A3 for a bipole with $d = 0.1$ mm and fit the slope. What happens to that slope with $d = 4$ mm?

## Part C · Recordings as spatial filters, with FFTs

The module writes the potential in spatial frequency $k$:
$\hat\phi(k) = \dfrac{e^{-|k|h}}{2\sigma|k|}\,\hat i_m(k)$. Distance from the tissue multiplies the spectrum by
$e^{-|k|h}$, a low-pass filter. A bipole multiplies it by $H(k) = 2i\sin(kd/2)$, a high-pass filter with nulls.

In [ ]:
L, n = 4096.0, 2**16
x = (np.arange(n) - n // 2) * (L / n)            # mm
k = 2 * np.pi * np.fft.fftfreq(n, d=L / n)       # rad/mm
im_hat = np.fft.fft(np.fft.ifftshift(membrane_current(x)))

def potential_at_height(hh):
    g = np.zeros_like(k)
    nz = k != 0
    g[nz] = np.exp(-np.abs(k[nz]) * hh) / (2 * sigma * np.abs(k[nz]))
    return np.fft.fftshift(np.real(np.fft.ifft(im_hat * g)))

phi_h = potential_at_height(h)
window = np.abs(x) < 20
fft_err = np.max(np.abs(phi_h[window] - eg.unipolar(x[window], a, A))) / np.ptp(phi_h)

fig, ax = plt.subplots()
pos = (k > 0) & (k < 6)
for hh in (0.5, 1.0, 2.0):
    spectrum = np.abs(np.fft.fft(np.fft.ifftshift(potential_at_height(hh))))
    ax.semilogy(k[pos], spectrum[pos], label=f"h = {hh} mm")
ax.set(xlabel="spatial frequency k (rad/mm)", ylabel="|spectrum| (relative)",
       title="Moving away from the tissue removes fine spatial detail")
ax.legend()
plt.show()
print(f"FFT route vs closed form: largest difference {fft_err:.1e} of peak-to-peak")

In [ ]:
assert fft_err < 1e-3

### C2 · A bipole in space equals a filter in frequency

Subtracting two shifted copies of the potential (space) and multiplying its spectrum by $H(k)$ (frequency) must give
the same signal.

In [ ]:
d = 2.0                                  # mm (m03 default)
direct = np.interp(x + d / 2, x, phi_h) - np.interp(x - d / 2, x, phi_h)
H = 2j * np.sin(k * d / 2)
via_fft = np.fft.fftshift(np.real(np.fft.ifft(np.fft.fft(np.fft.ifftshift(phi_h)) * H)))
mismatch = np.max(np.abs(direct[window] - via_fft[window])) / np.ptp(direct)

fig, ax = plt.subplots()
ax.plot(x[window], direct[window], lw=3, alpha=0.5, label="difference of two unipoles")
ax.plot(x[window], via_fft[window], "k--", label="spectrum × H(k)")
ax.set(xlabel="x (mm)", ylabel="bipolar (relative units)", title=f"Bipole with d = {d} mm, two ways")
ax.legend()
plt.show()
print(f"largest mismatch: {mismatch:.1e}")

In [ ]:
assert mismatch < 1e-9

**Your turn.** Where are the nulls of $H(k)$ for $d = 2$ mm? Build a potential that is a pure spatial cosine at the
first null and confirm the bipole records nothing. Then add disk averaging: multiply by $2J_1(kr)/(kr)$
(`eg.disk_transfer`) and see what a larger electrode does to the band the bipole passes.

## Part D · Fields, cliques and omnipoles

A clique of electrodes measures several bipoles at once. If the potential is nearly linear across the clique, each
bipole is a projection of the field $\mathbf E = -\nabla\phi$, so $\mathbf b = -D^{\mathsf T}\mathbf E$ and the
least-squares estimate is $\hat{\mathbf E} = -(DD^{\mathsf T})^{-1}D\,\mathbf b$. Reading $\hat{\mathbf E}$ along the
propagation direction gives the omnipolar signal.

In [ ]:
t = np.linspace(-40, 40, 4001)    # ms

def square_clique(d):
    positions = np.array([[0, 0], [d, 0], [0, d], [d, d]], dtype=float) - d / 2
    pairs = [(0, 1), (2, 3), (0, 2), (1, 3), (0, 3), (1, 2)]
    return positions, pairs

def clique_response(d, theta_deg, c=1.0):
    n = np.array([np.cos(np.deg2rad(theta_deg)), np.sin(np.deg2rad(theta_deg))])
    positions, pairs = square_clique(d)
    u = eg.unipolar(eg.front_distance(positions, n, t, c), a, A)
    E = eg.clique_field(positions, u, pairs)
    v, omega = eg.omnipolar(E, d)
    return n, u, E, v, omega

fig, axes = plt.subplots(1, 2, figsize=(8, 3.6))
for ax, d in zip(axes, (0.25, 2.0)):
    n, u, E, v, omega = clique_response(d, 30)
    ax.plot(E[0], E[1])
    ax.annotate("", xy=0.8 * np.abs(E).max() * n, xytext=(0, 0), arrowprops=dict(color="C3", width=2))
    ax.axline((0, 0), omega, ls="--", color="gray")
    ax.set(aspect="equal", xlabel="E_x", ylabel="E_y", title=f"field loop, d = {d} mm")
plt.tight_layout()
plt.show()

In [ ]:
directions = np.arange(0, 180, 10)

def omni_spread(d):
    p2p = [eg.peak_to_peak(clique_response(d, th)[3]) for th in directions]
    return max(p2p) / min(p2p) - 1

spreads = {d: omni_spread(d) for d in (0.25, 0.5, 1.0, 2.0, 4.0)}
for d, sp in spreads.items():
    print(f"d = {d} mm  (d/a = {d / a:.2f}): omnipole varies by {100 * sp:.1f}% across directions")

In [ ]:
assert spreads[0.25] < 0.02, "with spacing well below w + h the omnipole is nearly direction-independent"
assert list(spreads.values()) == sorted(spreads.values()), "the spread should grow with spacing"

### D2 · Conduction velocity from a single clique

Along the propagation direction the field is $-dU/ds$, and the unipolar slope is $-c\,dU/ds$. So the ratio of their
peak-to-peak amplitudes gives $c$ without measuring any activation times (module m25 returns to this).

In [ ]:
estimates = {}
for c_true in (0.5, 1.0, 1.5):
    n, u, E, v, omega = clique_response(0.25, 40, c=c_true)
    du_dt = np.gradient(u.mean(axis=0), t)
    estimates[c_true] = eg.peak_to_peak(du_dt) / eg.peak_to_peak(n @ E)
    print(f"true c = {c_true:.2f} m/s   estimated c = {estimates[c_true]:.3f} m/s")

In [ ]:
assert all(abs(est - c_true) / c_true < 0.02 for c_true, est in estimates.items())

**Your turn.** The estimate above uses the *true* direction `n`. Replace it with the loop direction `omega`. What
changes, and why does the sign of `omega` not matter here?

## Part E · Real bipolar recordings

The PhysioNet **Intracardiac Atrial Fibrillation Database** (doi:10.13026/C23S33) holds right-atrial recordings from
8 patients in atrial fibrillation or flutter. A decapolar catheter with 2-5-2 mm spacing gives 5 bipoles,
labeled CS12 (distal) to CS90 (proximal), 7 mm apart, sampled at 1 kHz with 3 surface ECG leads. PhysioNet notes that
**the amplitudes are uncalibrated**. Data license: Open Data Commons Attribution v1.0. Cite PhysioNet: Goldberger et al.,
*Circulation* 2000;101:e215 (doi:10.1161/01.cir.101.23.e215).

The next cell downloads one record (about 1 MB) and checks it against PhysioNet's published SHA-256 checksums.
Without internet access this part is skipped.

In [ ]:
from epml.data import DataUnavailable, fetch_iafdb

try:
    record_path = fetch_iafdb("iaf1_svc")
except DataUnavailable as err:
    record_path = None
    print("Part E skipped (no data):", err)

In [ ]:
if record_path:
    import wfdb
    rec = wfdb.rdrecord(record_path)
    print(f"sampling rate: {rec.fs} Hz, {rec.sig_len / rec.fs:.1f} s, signals: {rec.sig_name}")
    assert rec.fs == 1000
    bipoles = [i for i, name in enumerate(rec.sig_name) if name.upper().startswith("CS")]
    ecg = [i for i in range(rec.n_sig) if i not in bipoles][0]
    assert len(bipoles) == 5

    start, seconds = 0, 2.0
    seg = slice(int(start * rec.fs), int((start + seconds) * rec.fs))
    tt = np.arange(rec.sig_len)[seg] / rec.fs * 1000  # ms

    fig, ax = plt.subplots(figsize=(8, 5))
    for row, idx in enumerate(bipoles + [ecg]):
        trace = rec.p_signal[seg, idx]
        trace = (trace - np.mean(trace)) / (np.ptp(trace) + 1e-12)
        ax.plot(tt, trace - 1.2 * row, lw=0.8, color="k" if idx == ecg else "C0")
        ax.text(tt[0] - 40, -1.2 * row, rec.sig_name[idx], ha="right", va="center")
    ax.set(xlabel="time (ms)", yticks=[], title="Bipoles distal to proximal, then a surface ECG lead (each scaled to its own range)")
    plt.show()

### E2 · Delays between neighboring bipoles

If a wave travels along the catheter, each bipole fires a little after its neighbor. Cross-correlating neighboring
channels over a few seconds gives the typical delay. Divide the 7 mm spacing by that delay to get an *apparent*
velocity along the catheter.

In [ ]:
if record_path:
    from scipy.signal import correlate

    seg = slice(0, int(8 * rec.fs))
    max_lag = 40  # ms
    print("pair            delay (ms)   apparent velocity along catheter (m/s)")
    for i, j in zip(bipoles, bipoles[1:]):
        p = rec.p_signal[seg, i] - np.mean(rec.p_signal[seg, i])
        q = rec.p_signal[seg, j] - np.mean(rec.p_signal[seg, j])
        xc = correlate(q, p, mode="full")
        lags = np.arange(-len(p) + 1, len(p))
        keep = np.abs(lags) <= max_lag
        lag = lags[keep][np.argmax(xc[keep])] * 1000 / rec.fs
        speed = 7.0 / abs(lag) if lag else float("inf")
        print(f"{rec.sig_name[i]:>5} -> {rec.sig_name[j]:<5}  {lag:+8.0f}       {speed:6.2f}")

**What you cannot infer here, and why (connect to the module):**

- There are **no unipolar signals**, so you cannot separate near-field from far-field the way Part C does, or estimate
  a local field as in Part D.
- **Amplitudes are uncalibrated**, and bipolar amplitude depends on the angle between each bipole and the wave
  (Part B). A smaller signal on one channel does not, on its own, mean different tissue.
- The **apparent velocity** equals the true conduction velocity only if the wave travels along the catheter. A wave
  arriving at an angle makes neighboring bipoles fire closer together, so the apparent velocity is *too high*. During
  fibrillation the pattern also changes from beat to beat.

**Your turn.** Repeat E2 on several 2-second windows. How stable is the delay? Try another record, such as
`iaf1_afw` (catheter on the atrial free wall).

---

**Where next.** The module's derivations explain each result you reproduced here. Module m22 covers electrogram
annotation, and m25 covers conduction-velocity estimation in depth.